# Stage 2: Batch Transformation Pipeline — MediStream Telehealth
## ISM 6562 — Final Project
---
This notebook implements **Stage 2**: reading raw data from HDFS, cleaning it with PySpark, joining multiple sources, and writing curated/analytics Parquet outputs.

### Prerequisites
- HDFS cluster running with data in `/medistream/landing/`
- Spark cluster running (master + 2 workers + Jupyter)
- `docker compose up -d` and wait ~90s

## Part 0: Setup

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('MediStream-Stage2-BatchTransforms')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.pyspark.python', '/opt/miniconda/bin/python3')
    .config('spark.pyspark.driver.python', '/opt/conda/bin/python3')
    .getOrCreate())

print(f'Spark version:       {spark.version}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')
print(f'Application ID:      {spark.sparkContext.applicationId}')
print(f'Master URL:          {spark.sparkContext.master}')
print()
print('UIs:')
print('  Spark Master:  http://localhost:8080')
print('  Spark App UI:  http://localhost:4040')
print('  HDFS:          http://localhost:9870')

Spark version:       3.5.0
Default parallelism: 2
Application ID:      app-20260502032317-0001
Master URL:          spark://spark-master:7077

UIs:
  Spark Master:  http://localhost:8080
  Spark App UI:  http://localhost:4040
  HDFS:          http://localhost:9870


## Part 1: Ingest — Read Raw Data from HDFS Landing Zone

MediStream has 5 datasets: appointments (CSV), patient_vitals (JSON), session_quality (CSV), patient_feedback (JSON), physician_schedule (CSV). Spark reads `.gz` natively.

In [2]:
# Define paths — try HDFS first, fall back to local mount
HDFS_BASE = 'hdfs://namenode:9000/medistream/landing'
LOCAL_BASE = '/home/jovyan/data'

try:
    spark.read.csv(f'{HDFS_BASE}/appointments/', header=True, inferSchema=True).limit(1).count()
    BASE = HDFS_BASE
    print(f'Reading from HDFS: {HDFS_BASE}')
except Exception as e:
    BASE = LOCAL_BASE
    print(f'HDFS not available, reading from local mount: {LOCAL_BASE}')
    print(f'  (Error: {e})')

Reading from HDFS: hdfs://namenode:9000/medistream/landing


In [3]:
# 1.1 — Appointments (CSV)
appointments_raw = spark.read.csv(f'{BASE}/appointments/', header=True, inferSchema=True)
print(f'=== Appointments (raw) ===')
print(f'Rows: {appointments_raw.count():,}, Columns: {len(appointments_raw.columns)}')
appointments_raw.printSchema()
appointments_raw.show(5, truncate=False)

=== Appointments (raw) ===
Rows: 550,000, Columns: 10
root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- physician_id: string (nullable = true)
 |-- scheduled_time: timestamp (nullable = true)
 |-- actual_start: timestamp (nullable = true)
 |-- actual_end: timestamp (nullable = true)
 |-- specialty: string (nullable = true)
 |-- status: string (nullable = true)
 |-- wait_time_min: integer (nullable = true)
 |-- visit_type: string (nullable = true)

+--------------+----------+------------+-------------------+-------------------+-------------------+------------+---------+-------------+----------+
|appointment_id|patient_id|physician_id|scheduled_time     |actual_start       |actual_end         |specialty   |status   |wait_time_min|visit_type|
+--------------+----------+------------+-------------------+-------------------+-------------------+------------+---------+-------------+----------+
|APPT-0000001  |PAT-002222|DR-0596     |2024-12-02 08

In [4]:
# 1.2 — Patient Vitals (JSON)
vitals_raw = spark.read.json(f'{BASE}/patient_vitals/')
print(f'=== Patient Vitals (raw) ===')
print(f'Rows: {vitals_raw.count():,}, Columns: {len(vitals_raw.columns)}')
vitals_raw.printSchema()
vitals_raw.show(5, truncate=False)

=== Patient Vitals (raw) ===
Rows: 400,002, Columns: 8
root
 |-- _corrupt_record: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- measurement_type: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- reading: struct (nullable = true)
 |    |-- diastolic: long (nullable = true)
 |    |-- systolic: long (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)

+---------------+-----------+----------------+----------+---------+-------------------+-----+-----+
|_corrupt_record|device_type|measurement_type|patient_id|reading  |timestamp          |unit |value|
+---------------+-----------+----------------+----------+---------+-------------------+-----+-----+
|[              |NULL       |NULL            |NULL      |NULL     |NULL               |NULL |NULL |
|NULL           |wearable   |blood_glucose   |PAT-026907|NULL     |2024-04-06T12:53:00|mg/dL|203.0|
|NULL           |we

In [5]:
# 1.3 — Session Quality (CSV)
session_raw = spark.read.csv(f'{BASE}/session_quality/', header=True, inferSchema=True)
print(f'=== Session Quality (raw) ===')
print(f'Rows: {session_raw.count():,}, Columns: {len(session_raw.columns)}')
session_raw.printSchema()
session_raw.show(5, truncate=False)

=== Session Quality (raw) ===
Rows: 250,000, Columns: 9
root
 |-- session_id: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- video_resolution: string (nullable = true)
 |-- audio_quality_score: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- packet_loss_pct: double (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- os: string (nullable = true)

+------------+--------------+----------------+-------------------+----------+---------------+------------+-----------+-------+
|session_id  |appointment_id|video_resolution|audio_quality_score|latency_ms|packet_loss_pct|duration_sec|device_type|os     |
+------------+--------------+----------------+-------------------+----------+---------------+------------+-----------+-------+
|SESS-0000001|APPT-0000001  |480p            |7                  |43.8      |0.59           |396         |tablet     |macOS  |
|SESS-0000002|APPT-0000002  |48

In [6]:
# 1.4 — Patient Feedback (JSON)
feedback_raw = spark.read.json(f'{BASE}/patient_feedback/')
print(f'=== Patient Feedback (raw) ===')
print(f'Rows: {feedback_raw.count():,}, Columns: {len(feedback_raw.columns)}')
feedback_raw.printSchema()
feedback_raw.show(5, truncate=False)

=== Patient Feedback (raw) ===
Rows: 50,002, Columns: 7
root
 |-- _corrupt_record: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- categories_mentioned: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- feedback_id: string (nullable = true)
 |-- nps_score: long (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- rating: long (nullable = true)

+---------------+--------------+---------------------------------------------------+-----------+---------+----------+------+
|_corrupt_record|appointment_id|categories_mentioned                               |feedback_id|nps_score|patient_id|rating|
+---------------+--------------+---------------------------------------------------+-----------+---------+----------+------+
|[              |NULL          |NULL                                               |NULL       |NULL     |NULL      |NULL  |
|NULL           |APPT-0157393  |[]                                                 |FB

In [7]:
# 1.5 — Physician Schedule (CSV)
schedule_raw = spark.read.csv(f'{BASE}/physician_schedule/', header=True, inferSchema=True)
print(f'=== Physician Schedule (raw) ===')
print(f'Rows: {schedule_raw.count():,}, Columns: {len(schedule_raw.columns)}')
schedule_raw.printSchema()
schedule_raw.show(5, truncate=False)

=== Physician Schedule (raw) ===
Rows: 30,000, Columns: 7
root
 |-- physician_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- specialty: string (nullable = true)
 |-- max_appointments: integer (nullable = true)
 |-- actual_appointments: integer (nullable = true)

+------------+----------+-------------------+-------------------+-------------+----------------+-------------------+
|physician_id|date      |start_time         |end_time           |specialty    |max_appointments|actual_appointments|
+------------+----------+-------------------+-------------------+-------------+----------------+-------------------+
|DR-0058     |2024-11-02|2026-05-02 09:00:00|2026-05-02 20:00:00|primary_care |30              |12                 |
|DR-0564     |2024-01-10|2026-05-02 08:00:00|2026-05-02 19:00:00|allergy      |31              |23                 |
|DR-0656     |2024-08-07|2026-05-02 09:00:0

## Part 2: Curate — Clean, Standardize, Validate

For each dataset: drop nulls, cast types, trim strings, deduplicate, write Parquet to curated zone.

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, lower, to_date, to_timestamp, when

# Output paths
HDFS_CURATED = 'hdfs://namenode:9000/medistream/curated'
LOCAL_CURATED = '/home/jovyan/data/curated'
try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_CURATED}/_test')
    CURATED = HDFS_CURATED
    print(f'Writing curated to HDFS: {CURATED}')
except Exception:
    CURATED = LOCAL_CURATED
    print(f'Writing curated locally: {CURATED}')

Writing curated to HDFS: hdfs://namenode:9000/medistream/curated


In [9]:
# Helper: generic curation function
def curate_dataset(df, name, id_col=None):
    '''Clean a raw DataFrame: find PK, drop null PKs, dedup, trim strings, cast dates.'''
    # Find primary key
    if id_col is None:
        id_col = next((c for c in df.columns if 'id' in c.lower()), df.columns[0])
    print(f'--- Curating {name} (PK: {id_col}) ---')
    
    # Drop null PKs
    df = df.filter(col(id_col).isNotNull())
    
    # Dedup
    before = df.count()
    df = df.dropDuplicates([id_col])
    after = df.count()
    print(f'  Dedup: {before} -> {after} ({before - after} removed)')
    
    # Trim strings
    for c_name, c_type in df.dtypes:
        if c_type == 'string':
            df = df.withColumn(c_name, trim(col(c_name)))
    
    # Cast date/time string columns
    for c_name in df.columns:
        if ('date' in c_name.lower() or 'time' in c_name.lower()) and dict(df.dtypes).get(c_name) == 'string':
            df = df.withColumn(c_name, to_timestamp(col(c_name)))
    
    return df

# Helper: flatten nested JSON structs
def flatten_structs(df):
    struct_cols = [c for c, t in df.dtypes if t.startswith('struct')]
    for sc in struct_cols:
        for sf in df.select(f'{sc}.*').columns:
            df = df.withColumn(f'{sc}_{sf}', col(f'{sc}.{sf}'))
        df = df.drop(sc)
    return df

In [10]:
# 2.1 Curate Appointments
appointments = curate_dataset(appointments_raw, 'appointments')
appointments.write.mode('overwrite').parquet(f'{CURATED}/appointments')
print(f'  Written: {appointments.count():,} rows')
appointments.printSchema()
appointments.show(5, truncate=False)

--- Curating appointments (PK: appointment_id) ---
  Dedup: 550000 -> 550000 (0 removed)
  Written: 550,000 rows
root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- physician_id: string (nullable = true)
 |-- scheduled_time: timestamp (nullable = true)
 |-- actual_start: timestamp (nullable = true)
 |-- actual_end: timestamp (nullable = true)
 |-- specialty: string (nullable = true)
 |-- status: string (nullable = true)
 |-- wait_time_min: integer (nullable = true)
 |-- visit_type: string (nullable = true)

+--------------+----------+------------+-------------------+-------------------+-------------------+----------------+---------+-------------+----------+
|appointment_id|patient_id|physician_id|scheduled_time     |actual_start       |actual_end         |specialty       |status   |wait_time_min|visit_type|
+--------------+----------+------------+-------------------+-------------------+-------------------+----------------+---------+--------

In [11]:
# 2.2 Curate Patient Vitals (JSON — may have nested structs)
vitals_flat = flatten_structs(vitals_raw)
vitals = curate_dataset(vitals_flat, 'patient_vitals')
vitals.write.mode('overwrite').parquet(f'{CURATED}/patient_vitals')
print(f'  Written: {vitals.count():,} rows')
vitals.printSchema()
vitals.show(5, truncate=False)

--- Curating patient_vitals (PK: patient_id) ---
  Dedup: 400000 -> 49984 (350016 removed)
  Written: 49,984 rows
root
 |-- _corrupt_record: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- measurement_type: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: double (nullable = true)
 |-- reading_diastolic: long (nullable = true)
 |-- reading_systolic: long (nullable = true)

+---------------+-----------+----------------+----------+-------------------+----------+-----+-----------------+----------------+
|_corrupt_record|device_type|measurement_type|patient_id|timestamp          |unit      |value|reading_diastolic|reading_systolic|
+---------------+-----------+----------------+----------+-------------------+----------+-----+-----------------+----------------+
|NULL           |clinic     |weight          |PAT-000001|2024-03-30 17:49:00|lbs       |240.5|NULL  

In [12]:
# 2.3 Curate Session Quality
session = curate_dataset(session_raw, 'session_quality')
session.write.mode('overwrite').parquet(f'{CURATED}/session_quality')
print(f'  Written: {session.count():,} rows')
session.printSchema()
session.show(5, truncate=False)

--- Curating session_quality (PK: session_id) ---
  Dedup: 250000 -> 250000 (0 removed)
  Written: 250,000 rows
root
 |-- session_id: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- video_resolution: string (nullable = true)
 |-- audio_quality_score: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- packet_loss_pct: double (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- os: string (nullable = true)

+------------+--------------+----------------+-------------------+----------+---------------+------------+-----------+--------+
|session_id  |appointment_id|video_resolution|audio_quality_score|latency_ms|packet_loss_pct|duration_sec|device_type|os      |
+------------+--------------+----------------+-------------------+----------+---------------+------------+-----------+--------+
|SESS-0000002|APPT-0000002  |480p            |4                  |180.5     |6.46           |975   

In [13]:
# 2.4 Curate Patient Feedback (JSON — may have nested structs)
feedback_flat = flatten_structs(feedback_raw)
feedback = curate_dataset(feedback_flat, 'patient_feedback')
feedback.write.mode('overwrite').parquet(f'{CURATED}/patient_feedback')
print(f'  Written: {feedback.count():,} rows')
feedback.printSchema()
feedback.show(5, truncate=False)

--- Curating patient_feedback (PK: appointment_id) ---
  Dedup: 50000 -> 46063 (3937 removed)
  Written: 46,063 rows
root
 |-- _corrupt_record: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- categories_mentioned: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- feedback_id: string (nullable = true)
 |-- nps_score: long (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- rating: long (nullable = true)

+---------------+--------------+---------------------------------------------------+-----------+---------+----------+------+
|_corrupt_record|appointment_id|categories_mentioned                               |feedback_id|nps_score|patient_id|rating|
+---------------+--------------+---------------------------------------------------+-----------+---------+----------+------+
|NULL           |APPT-0000003  |[]                                                 |FB-020310  |9        |PAT-021035|4     |
|NULL           |APPT-000

In [14]:
# 2.5 Curate Physician Schedule
schedule = curate_dataset(schedule_raw, 'physician_schedule')
schedule.write.mode('overwrite').parquet(f'{CURATED}/physician_schedule')
print(f'  Written: {schedule.count():,} rows')
schedule.printSchema()
schedule.show(5, truncate=False)

--- Curating physician_schedule (PK: physician_id) ---
  Dedup: 30000 -> 800 (29200 removed)
  Written: 800 rows
root
 |-- physician_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- specialty: string (nullable = true)
 |-- max_appointments: integer (nullable = true)
 |-- actual_appointments: integer (nullable = true)

+------------+----------+-------------------+-------------------+----------------+----------------+-------------------+
|physician_id|date      |start_time         |end_time           |specialty       |max_appointments|actual_appointments|
+------------+----------+-------------------+-------------------+----------------+----------------+-------------------+
|DR-0001     |2024-10-30|2026-05-02 08:00:00|2026-05-02 16:00:00|orthopedics     |19              |18                 |
|DR-0002     |2024-12-09|2026-05-02 09:00:00|2026-05-02 17:00:00|dermatology     |22         

## Part 3: Analytics — Joins, Aggregations, Business Questions

Read from curated zone and produce 5 analytics outputs:
1. **No-Show Features** — appointments + vitals + session quality
2. **Physician Performance** — appointments + feedback + schedule
3. **Session Quality Agg** — session quality + appointments
4. **Utilization Rates** — schedule + appointments
5. **Patient Journey** — appointments + vitals + feedback

In [15]:
# Read curated data
appointments_c = spark.read.parquet(f'{CURATED}/appointments')
vitals_c       = spark.read.parquet(f'{CURATED}/patient_vitals')
session_c      = spark.read.parquet(f'{CURATED}/session_quality')
feedback_c     = spark.read.parquet(f'{CURATED}/patient_feedback')
schedule_c     = spark.read.parquet(f'{CURATED}/physician_schedule')

print('Curated data loaded:')
for name, df in [('appointments', appointments_c), ('vitals', vitals_c),
                 ('session_quality', session_c), ('feedback', feedback_c),
                 ('schedule', schedule_c)]:
    print(f'  {name:<20s}  {df.count():>8,} rows, {len(df.columns):>3} cols')

# Analytics output path
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'
try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_test')
    ANALYTICS = HDFS_ANALYTICS
except Exception:
    ANALYTICS = LOCAL_ANALYTICS
print(f'Analytics output: {ANALYTICS}')

Curated data loaded:
  appointments           550,000 rows,  10 cols
  vitals                  49,984 rows,   9 cols
  session_quality        250,000 rows,   9 cols
  feedback                46,063 rows,   7 cols
  schedule                   800 rows,   7 cols
Analytics output: hdfs://namenode:9000/medistream/analytics


In [16]:
# Helper: find common join keys between two DataFrames
import itertools

def find_join_keys(df1, df2):
    cols1 = {c.lower(): c for c in df1.columns}
    cols2 = {c.lower(): c for c in df2.columns}
    common = set(cols1.keys()) & set(cols2.keys())
    id_keys = [cols1[k] for k in common if 'id' in k]
    return id_keys if id_keys else [cols1[k] for k in common]

def find_col(df, *keywords):
    '''Find a column containing any of the keywords.'''
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

# Show join keys between all dataset pairs
datasets = {'appointments': appointments_c, 'vitals': vitals_c,
            'session': session_c, 'feedback': feedback_c, 'schedule': schedule_c}
print('Join keys between datasets:')
for (n1, d1), (n2, d2) in itertools.combinations(datasets.items(), 2):
    keys = find_join_keys(d1, d2)
    if keys:
        print(f'  {n1} <-> {n2}: {keys}')

Join keys between datasets:
  appointments <-> vitals: ['patient_id']
  appointments <-> session: ['appointment_id']
  appointments <-> feedback: ['patient_id', 'appointment_id']
  appointments <-> schedule: ['physician_id']
  vitals <-> session: ['device_type']
  vitals <-> feedback: ['patient_id']
  session <-> feedback: ['appointment_id']


### 3.1 — No-Show Feature Engineering
Join appointments + vitals + session quality to build features for predicting no-shows.

In [17]:
# JOIN 1: appointments + vitals
jk_av = find_join_keys(appointments_c, vitals_c)
print(f'Join keys (appt <-> vitals): {jk_av}')

if jk_av:
    jk = jk_av[0]
    # Aggregate vitals per join key to avoid row explosion
    vital_nums = [c for c, t in vitals_c.dtypes if t in ('double','float','int','bigint','long') and 'id' not in c.lower()]
    v_aggs = [F.count('*').alias('vital_reading_count')]
    for vc in vital_nums[:5]:
        v_aggs.append(F.round(F.avg(col(vc)), 2).alias(f'avg_{vc}'))
        v_aggs.append(F.round(F.stddev(col(vc)), 2).alias(f'std_{vc}'))
    vitals_summary = vitals_c.groupBy(jk).agg(*v_aggs)
    no_show = appointments_c.join(vitals_summary, on=jk, how='left')
else:
    no_show = appointments_c

# JOIN 2: + session quality
jk_as = find_join_keys(appointments_c, session_c)
print(f'Join keys (appt <-> session): {jk_as}')
if jk_as:
    sk = jk_as[0]
    sess_nums = [c for c, t in session_c.dtypes if t in ('double','float','int','bigint','long') and 'id' not in c.lower()]
    s_aggs = [F.count('*').alias('session_count')]
    for sc in sess_nums[:3]:
        s_aggs.append(F.round(F.avg(col(sc)), 2).alias(f'avg_sess_{sc}'))
    sess_summary = session_c.groupBy(sk).agg(*s_aggs)
    no_show = no_show.join(sess_summary, on=sk, how='left')

no_show.write.mode('overwrite').parquet(f'{ANALYTICS}/no_show_features')
print(f'\nNo-Show Features: {no_show.count():,} rows, {len(no_show.columns)} cols')
no_show.printSchema()
no_show.show(5, truncate=False)

Join keys (appt <-> vitals): ['patient_id']
Join keys (appt <-> session): ['appointment_id']

No-Show Features: 550,000 rows, 21 cols
root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- physician_id: string (nullable = true)
 |-- scheduled_time: timestamp (nullable = true)
 |-- actual_start: timestamp (nullable = true)
 |-- actual_end: timestamp (nullable = true)
 |-- specialty: string (nullable = true)
 |-- status: string (nullable = true)
 |-- wait_time_min: integer (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- vital_reading_count: long (nullable = true)
 |-- avg_value: double (nullable = true)
 |-- std_value: double (nullable = true)
 |-- avg_reading_diastolic: double (nullable = true)
 |-- std_reading_diastolic: double (nullable = true)
 |-- avg_reading_systolic: double (nullable = true)
 |-- std_reading_systolic: double (nullable = true)
 |-- session_count: long (nullable = true)
 |-- avg_sess_audio_quality_score: do

### 3.2 — Physician Performance
Join appointments + feedback + schedule to compute per-physician KPIs.

In [18]:
phys_col_a = find_col(appointments_c, 'physician', 'doctor', 'provider')
phys_col_f = find_col(feedback_c, 'physician', 'doctor', 'provider')
phys_col_s = find_col(schedule_c, 'physician', 'doctor', 'provider')
status_col = find_col(appointments_c, 'status', 'no_show', 'noshow')
rating_col = find_col(feedback_c, 'rating', 'score', 'satisfaction')

print(f'Physician col in appointments: {phys_col_a}')
print(f'Physician col in feedback:     {phys_col_f}')
print(f'Physician col in schedule:     {phys_col_s}')
print(f'Status col: {status_col}, Rating col: {rating_col}')

if status_col:
    print('\nAppointment status values:')
    appointments_c.groupBy(status_col).count().show()

Physician col in appointments: physician_id
Physician col in feedback:     None
Physician col in schedule:     physician_id
Status col: status, Rating col: nps_score

Appointment status values:
+---------+------+
|   status| count|
+---------+------+
|completed|396186|
|  no_show| 99178|
|cancelled| 54636|
+---------+------+



In [19]:
if phys_col_a:
    # Base: appointments per physician
    appt_aggs = [F.count('*').alias('total_appointments')]
    if status_col:
        appt_aggs.append(
            F.round(F.sum(F.when(
                (F.lower(col(status_col)).contains('no')) |
                (F.lower(col(status_col)).contains('cancel')),
                1).otherwise(0)) / F.count('*') * 100, 1
            ).alias('no_show_rate_pct'))
    phys_perf = appointments_c.groupBy(phys_col_a).agg(*appt_aggs)

    # JOIN feedback
    if phys_col_f and rating_col:
        fb_stats = feedback_c.groupBy(phys_col_f).agg(
            F.round(F.avg(col(rating_col)), 2).alias('avg_patient_rating'),
            F.count('*').alias('feedback_count'))
        if phys_col_f != phys_col_a:
            fb_stats = fb_stats.withColumnRenamed(phys_col_f, phys_col_a)
        phys_perf = phys_perf.join(fb_stats, on=phys_col_a, how='left')

    # JOIN schedule
    if phys_col_s:
        sched_stats = schedule_c.groupBy(phys_col_s).agg(F.count('*').alias('scheduled_slots'))
        if phys_col_s != phys_col_a:
            sched_stats = sched_stats.withColumnRenamed(phys_col_s, phys_col_a)
        phys_perf = phys_perf.join(sched_stats, on=phys_col_a, how='left')
        phys_perf = phys_perf.withColumn('utilization_pct',
            F.round(col('total_appointments') / col('scheduled_slots') * 100, 1))

    phys_perf = phys_perf.orderBy(F.desc('total_appointments'))
    phys_perf.write.mode('overwrite').parquet(f'{ANALYTICS}/physician_perf')
    print(f'Physician Performance: {phys_perf.count():,} rows')
    phys_perf.printSchema()
    phys_perf.show(10, truncate=False)
else:
    print('No physician column found — skipping')

Physician Performance: 800 rows
root
 |-- physician_id: string (nullable = true)
 |-- total_appointments: long (nullable = false)
 |-- no_show_rate_pct: double (nullable = true)
 |-- scheduled_slots: long (nullable = true)
 |-- utilization_pct: double (nullable = true)

+------------+------------------+----------------+---------------+---------------+
|physician_id|total_appointments|no_show_rate_pct|scheduled_slots|utilization_pct|
+------------+------------------+----------------+---------------+---------------+
|DR-0497     |776               |27.8            |1              |77600.0        |
|DR-0333     |767               |26.5            |1              |76700.0        |
|DR-0331     |764               |28.1            |1              |76400.0        |
|DR-0001     |759               |27.9            |1              |75900.0        |
|DR-0157     |759               |28.6            |1              |75900.0        |
|DR-0033     |755               |31.0            |1              

### 3.3 — Session Quality Aggregations
Join session quality + appointments, aggregate by grouping column.

In [20]:
sq_keys = find_join_keys(session_c, appointments_c)
if sq_keys:
    session_enriched = session_c.join(appointments_c, on=sq_keys[0], how='inner')
else:
    session_enriched = session_c

quality_cols = [c for c, t in session_c.dtypes if t in ('double','float','int','bigint','long') and 'id' not in c.lower()]
print(f'Quality metric columns: {quality_cols}')

# Find grouping column
group_col = None
for kw in ['specialty', 'department', 'dept', 'type']:
    group_col = find_col(session_enriched, kw)
    if group_col: break
if not group_col:
    group_col = find_col(session_enriched, 'physician', 'doctor', 'provider')

sq_aggs = [F.count('*').alias('session_count')]
for qc in quality_cols[:4]:
    sq_aggs.append(F.round(F.avg(col(qc)), 2).alias(f'avg_{qc}'))
    sq_aggs.append(F.round(F.min(col(qc)), 2).alias(f'min_{qc}'))
    sq_aggs.append(F.round(F.max(col(qc)), 2).alias(f'max_{qc}'))

if group_col:
    print(f'Grouping by: {group_col}')
    sq_agg = session_enriched.groupBy(group_col).agg(*sq_aggs).orderBy(F.desc('session_count'))
else:
    sq_agg = session_enriched.agg(*sq_aggs)

sq_agg.write.mode('overwrite').parquet(f'{ANALYTICS}/session_quality_agg')
print(f'Session Quality Agg: {sq_agg.count():,} rows')
sq_agg.show(15, truncate=False)

Quality metric columns: ['audio_quality_score', 'latency_ms', 'packet_loss_pct', 'duration_sec']
Grouping by: specialty
Session Quality Agg: 15 rows
+----------------+-------------+-----------------------+-----------------------+-----------------------+--------------+--------------+--------------+-------------------+-------------------+-------------------+----------------+----------------+----------------+
|specialty       |session_count|avg_audio_quality_score|min_audio_quality_score|max_audio_quality_score|avg_latency_ms|min_latency_ms|max_latency_ms|avg_packet_loss_pct|min_packet_loss_pct|max_packet_loss_pct|avg_duration_sec|min_duration_sec|max_duration_sec|
+----------------+-------------+-----------------------+-----------------------+-----------------------+--------------+--------------+--------------+-------------------+-------------------+-------------------+----------------+----------------+----------------+
|primary_care    |60559        |7.55                   |4           

### 3.4 — Utilization Rates
Join physician schedule + appointments to compute slot fill rates.

In [21]:
if phys_col_a and phys_col_s:
    date_col_a = find_col(appointments_c, 'date')
    date_col_s = find_col(schedule_c, 'date')

    if date_col_a and date_col_s:
        appt_daily = appointments_c.groupBy(
            col(phys_col_a), F.to_date(col(date_col_a)).alias('the_date')
        ).agg(F.count('*').alias('filled'))
        sched_daily = schedule_c.groupBy(
            col(phys_col_s).alias(phys_col_a), F.to_date(col(date_col_s)).alias('the_date')
        ).agg(F.count('*').alias('available'))
        util = sched_daily.join(appt_daily, on=[phys_col_a, 'the_date'], how='left').fillna(0, ['filled'])
        util = util.withColumn('utilization_pct', F.round(col('filled')/col('available')*100, 1))
    else:
        appt_t = appointments_c.groupBy(phys_col_a).agg(F.count('*').alias('filled'))
        sched_t = schedule_c.groupBy(col(phys_col_s).alias(phys_col_a)).agg(F.count('*').alias('available'))
        util = sched_t.join(appt_t, on=phys_col_a, how='left').fillna(0)
        util = util.withColumn('utilization_pct', F.round(col('filled')/col('available')*100, 1))

    util = util.orderBy(F.desc('utilization_pct'))
    util.write.mode('overwrite').parquet(f'{ANALYTICS}/utilization_rates')
    print(f'Utilization Rates: {util.count():,} rows')
    util.show(10, truncate=False)
else:
    print('Missing physician columns — cannot compute utilization')

Utilization Rates: 800 rows
+------------+---------+------+---------------+
|physician_id|available|filled|utilization_pct|
+------------+---------+------+---------------+
|DR-0497     |1        |776   |77600.0        |
|DR-0333     |1        |767   |76700.0        |
|DR-0331     |1        |764   |76400.0        |
|DR-0001     |1        |759   |75900.0        |
|DR-0157     |1        |759   |75900.0        |
|DR-0033     |1        |755   |75500.0        |
|DR-0435     |1        |753   |75300.0        |
|DR-0491     |1        |752   |75200.0        |
|DR-0163     |1        |751   |75100.0        |
|DR-0385     |1        |746   |74600.0        |
+------------+---------+------+---------------+
only showing top 10 rows



### 3.5 — Patient Journey
Join appointments + vitals + feedback to build an end-to-end view per patient.

In [22]:
pat_a = find_col(appointments_c, 'patient_id', 'patient')
pat_v = find_col(vitals_c, 'patient_id', 'patient')
pat_f = find_col(feedback_c, 'patient_id', 'patient')
print(f'Patient col — appt: {pat_a}, vitals: {pat_v}, feedback: {pat_f}')

if pat_a:
    date_c = find_col(appointments_c, 'date', 'time', 'scheduled')
    print(f'Date column found: {date_c}')
    
    if date_c:
        journey = appointments_c.groupBy(pat_a).agg(
            F.count('*').alias('total_appointments'),
            F.min(col(date_c)).alias('first_appt'),
            F.max(col(date_c)).alias('last_appt'))
    else:
        journey = appointments_c.groupBy(pat_a).agg(
            F.count('*').alias('total_appointments'),
            F.lit(None).cast('timestamp').alias('first_appt'),
            F.lit(None).cast('timestamp').alias('last_appt'))

    # Join vitals summary
    if pat_v:
        v_nums = [c for c, t in vitals_c.dtypes if t in ('double','float','int','bigint','long') and 'id' not in c.lower()]
        va = [F.count('*').alias('vital_readings')]
        for vc in v_nums[:3]:
            va.append(F.round(F.avg(col(vc)), 2).alias(f'avg_{vc}'))
        vpp = vitals_c.groupBy(pat_v).agg(*va)
        if pat_v != pat_a: vpp = vpp.withColumnRenamed(pat_v, pat_a)
        journey = journey.join(vpp, on=pat_a, how='left')

    # Join feedback summary
    if pat_f:
        rc = find_col(feedback_c, 'rating', 'score', 'satisfaction')
        fa = [F.count('*').alias('feedback_count')]
        if rc: fa.append(F.round(F.avg(col(rc)), 2).alias('avg_satisfaction'))
        fpp = feedback_c.groupBy(pat_f).agg(*fa)
        if pat_f != pat_a: fpp = fpp.withColumnRenamed(pat_f, pat_a)
        journey = journey.join(fpp, on=pat_a, how='left')

    journey = journey.orderBy(F.desc('total_appointments'))
    journey.write.mode('overwrite').parquet(f'{ANALYTICS}/patient_journey')
    print(f'Patient Journey: {journey.count():,} rows, {len(journey.columns)} cols')
    journey.printSchema()
    journey.show(10, truncate=False)
else:
    print('No patient column found')

Patient col — appt: patient_id, vitals: patient_id, feedback: patient_id
Date column found: scheduled_time
Patient Journey: 50,000 rows, 10 cols
root
 |-- patient_id: string (nullable = true)
 |-- total_appointments: long (nullable = false)
 |-- first_appt: timestamp (nullable = true)
 |-- last_appt: timestamp (nullable = true)
 |-- vital_readings: long (nullable = true)
 |-- avg_value: double (nullable = true)
 |-- avg_reading_diastolic: double (nullable = true)
 |-- avg_reading_systolic: double (nullable = true)
 |-- feedback_count: long (nullable = true)
 |-- avg_satisfaction: double (nullable = true)

+----------+------------------+-------------------+-------------------+--------------+---------+---------------------+--------------------+--------------+----------------+
|patient_id|total_appointments|first_appt         |last_appt          |vital_readings|avg_value|avg_reading_diastolic|avg_reading_systolic|feedback_count|avg_satisfaction|
+----------+------------------+------------

## Part 4: Verification

In [23]:
print('=' * 60)
print('CURATED ZONE')
print('=' * 60)
for n in ['appointments','patient_vitals','session_quality','patient_feedback','physician_schedule']:
    try:
        df = spark.read.parquet(f'{CURATED}/{n}')
        print(f'  {n:<25s} {df.count():>8,} rows, {len(df.columns):>3} cols  OK')
    except Exception as e:
        print(f'  {n:<25s} ERROR: {e}')

print()
print('=' * 60)
print('ANALYTICS ZONE')
print('=' * 60)
for n in ['no_show_features','physician_perf','session_quality_agg','utilization_rates','patient_journey']:
    try:
        df = spark.read.parquet(f'{ANALYTICS}/{n}')
        print(f'  {n:<25s} {df.count():>8,} rows, {len(df.columns):>3} cols  OK')
    except Exception as e:
        print(f'  {n:<25s} ERROR: {e}')

CURATED ZONE
  appointments               550,000 rows,  10 cols  OK
  patient_vitals              49,984 rows,   9 cols  OK
  session_quality            250,000 rows,   9 cols  OK
  patient_feedback            46,063 rows,   7 cols  OK
  physician_schedule             800 rows,   7 cols  OK

ANALYTICS ZONE
  no_show_features           550,000 rows,  21 cols  OK
  physician_perf                 800 rows,   5 cols  OK
  session_quality_agg             15 rows,  14 cols  OK
  utilization_rates              800 rows,   4 cols  OK
  patient_journey             50,000 rows,  10 cols  OK


In [24]:
print('Open http://localhost:4040/SQL/ to see the DAG for the queries above.')
print()
print('Stage 2 complete.')
print('  - 5 datasets curated (Parquet)')
print('  - 5 analytics outputs with joins across multiple sources')
print('  - Next: Stage 3 (Kafka streaming), Stage 4 (Airflow orchestration)')

Open http://localhost:4040/SQL/ to see the DAG for the queries above.

Stage 2 complete.
  - 5 datasets curated (Parquet)
  - 5 analytics outputs with joins across multiple sources
  - Next: Stage 3 (Kafka streaming), Stage 4 (Airflow orchestration)


In [25]:
# Uncomment to stop the SparkSession when done
# spark.stop()